### Define the model and API to use. You need an API key here.

In [258]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
from statistics import mean
import json, re, ast

load_dotenv()

# This will look for "ANTHROPIC_API_KEY=" in a .env file.
# Go to Claude console, generate an API, and purchase some credits.
client = Anthropic()
model = "claude-haiku-4-5"

### Create the helper functions.

In [346]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 4096,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

### Generate the dataset

In [389]:
# Function to generate a new dataset
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing a task that requires Python, JSON, or Regex to complete.

Example output:
```json
[
    {
        "task": "Write a Python function get_s3_bucket_names_with_prefix(prefix: str) -> list[str] that uses AWS S3 to return bucket names starting with the given prefix.",
        "format": "python",
        "expected_output": "def get_s3_bucket_names_with_prefix(prefix: str) -> list[str]:\\n import boto3\\n s3 = boto3.client('s3')\\n response = s3.list_buckets()\\n return [bucket['Name'] for bucket in response.get('Buckets', []) if bucket['Name'].startswith(prefix)]",
        "solution_criteria": [
            "Defines a function named get_s3_bucket_names_with_prefix",
            "Accepts one parameter named prefix with type str",
            "Returns a list[str]", "Uses boto3.client('s3')",
            "Calls list_buckets",
            "Filters bucket names using startswith(prefix)"
        ],
        "common_mistakes": [
            "Returns full bucket objects instead of bucket names",
            "Does not filter by the prefix parameter",
            "Uses a hard-coded prefix"
        ]
    },
    {
        "task": "Create a JSON IAM policy object that allows lambda:InvokeFunction only on the AWS Lambda function arn:aws:lambda:us-east-1:123456789012:function:ProcessOrders.",
        "format": "json",
        "expected_output": "{\\"Version\\":\\"2012-10-17\\",\\"Statement\\":[{\\"Effect\\":\\"Allow\\",\\"Action\\":\\"lambda:InvokeFunction\\",\\"Resource\\":\\"arn:aws:lambda:us-east-1:123456789012:function:ProcessOrders\\"}]}",
        "solution_criteria": [
            "Contains Version set to 2012-10-17",
            "Contains one Statement object",
            "Statement Effect is Allow",
            "Statement Action is lambda:InvokeFunction",
            "Statement Resource is arn:aws:lambda:us-east-1:123456789012:function:ProcessOrders" ],
        "common_mistakes": [
            "Uses wildcard resource",
            "Uses lambda:* instead of lambda:InvokeFunction",
            "Omits the Version field"
        ]
    },
    {
        "task": "Write a regex that matches AWS EC2 instance IDs that start with i- followed by exactly 17 lowercase hexadecimal characters.",
        "format": "regex",
        "expected_output": "^i-[0-9a-f]{17}$",
        "solution_criteria": [
            "Pattern starts with ^",
            "Pattern ends with $",
            "Matches the literal prefix i-",
            "Requires exactly 17 characters after i-",
            "Allows only lowercase hexadecimal characters 0-9 and a-f" ],
        "common_mistakes": [
            "Allows uppercase hexadecimal characters",
            "Allows fewer or more than 17 characters",
            "Does not anchor the pattern"
        ]
    }
]
```

Generate exactly 3 JSON objects
* Exactly 1 python task
* Exactly 1 json task
* Exactly 1 regex task

"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
   # print(repr(text))
    return json.loads(text)

In [390]:
# Generate the dataset and write it to 'dataset.json'
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

### Create the model grading pipeline.

In [391]:
# Function to grade a test case + output using a model

def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

### Pass a test case to Claude

In [392]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

### Code grader

In [393]:
# Functions to validate the output structure

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


### Model grader and combine the Model and Code scores.

In [394]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [395]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [396]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.166666666666666


In [364]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport boto3\nfrom typing import Any\n\n\ndef list_ec2_instances_by_tag(tag_key: str, tag_value: str) -> list[dict]:\n    ec2_client = boto3.client('ec2')\n    \n    response = ec2_client.describe_instances(\n        Filters=[\n            {\n                'Name': f'tag:{tag_key}',\n                'Values': [tag_value]\n            }\n        ]\n    )\n    \n    instances = []\n    for reservation in response['Reservations']:\n        for instance in reservation['Instances']:\n            instances.append({\n                'InstanceId': instance['InstanceId'],\n                'State': instance['State']['Name'],\n                'InstanceType': instance['InstanceType']\n            })\n    \n    return instances\n",
    "test_case": {
      "task": "Write a Python function list_ec2_instances_by_tag(tag_key: str, tag_value: str) -> list[dict] that queries AWS EC2 instances filtered by a specific tag and returns a list of instance dictionaries containing instan